# Treinar detector de pads (YOLO11)

1. Faça upload do dataset (zip) gerado por `tools/exportar_dataset.py`
2. Rode as células na ordem
3. Baixe `pads.onnx` e salve em `anotador-modular/models/pads.onnx`

No app: `Ctrl+K` → **Detector IA (modelo ONNX)**.

In [ ]:
!pip install -q ultralytics onnx==1.16.2
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Upload do zip do dataset
from google.colab import files
import os
up = files.upload()
for nome in up:
    os.makedirs('/content/dataset', exist_ok=True)
    os.system(f'unzip -q -o "{nome}" -d /content/dataset')
print(os.listdir('/content/dataset')[:10])

In [ ]:
# Ajuste o caminho do data.yaml se necessário
YAML = '/content/dataset/dataset_placas/data.yaml'
print(open(YAML).read())

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11n.pt')
result = modelo.train(data=YAML, epochs=80, imgsz=1024, batch=16, patience=25,
                      project='/content/runs', name='pads', exist_ok=True)

In [ ]:
# Exporta ONNX e baixa
onnx_path = modelo.export(format='onnx', imgsz=1024, opset=12, simplify=True)
print(onnx_path)
from google.colab import files
files.download(str(onnx_path))

In [ ]:
# Opcional: validar rapidamente
metrics = modelo.val(data=YAML, imgsz=1024)
print(metrics.box.map50, metrics.box.map)